# Splitting the evaluation set into val and test (KAN-53)
Gözde Nur Özdemir · ELIZA · Sprint 2

**Why two splits?** We choose the search settings (word removal, search mode, embedding model, top-k) by looking at scores. If the same questions are used for the final score, the score is too optimistic, because the settings were chosen to fit them. So one part of the questions, **`test`**, is put aside and frozen: it is used only for the final evaluation. All tuning uses **`val`**.

**Rules** (KAN-53 and the D2 acceptance criteria in the charter):
1. At least **one third** of the questions are `test`.
2. At least **one question of each safety category** (out of scope, insufficient evidence, legal decision, adversarial) is in `test`.
3. A **follow-up question** stays in the same split as the question it follows (column `context`, e.g. *"follows B-01"*).
4. Every question has a **unique ID**.
5. Questions **already used to choose settings** stay in `val`: B-01 to B-06 were used in `src/evaluation.ipynb`.
6. Each use case (a)–(d) gets at least one `test` question, so the final score covers all four.

The choice is random with a fixed seed, so running the notebook again gives the same split.

**When:** run once on 11/11/2026, after every member has added their questions to `docs/eval/eval-set-v1.csv`. It writes the `split` column and saves `eval-set-v1-val.csv` and `eval-set-v1-test.csv`; the test file is then frozen.

In [ ]:
import csv, hashlib, math, os, random, re
import pandas as pd
random.seed(42)

FOLDER = "../docs/eval/" if os.path.exists("../docs/eval/eval-set-v1.csv") else ""   # in the repository, or the file next to the notebook
questions = pd.read_csv(FOLDER + "eval-set-v1.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
assert questions["id"].is_unique, "Rule 4: two questions have the same ID: " + str(questions[questions["id"].duplicated()]["id"].tolist())
print(len(questions), "questions")
questions[["id", "type", "category", "use_case", "context"]]

## 1. Put follow-up questions together with the question they follow (rule 3)
A **group** is a question together with its follow-ups. Groups are moved to `val` or `test` as one piece.

In [ ]:
follows = {r["id"]: m[1] for _, r in questions.iterrows() if (m := re.search(r"follows ([A-Z]+-[A-Z0-9-]+)", r["context"]))}
group_of = {}
for qid in questions["id"]:
    first = qid
    while first in follows:              # go back to the first question of the chain
        first = follows[first]
    group_of[qid] = first
questions["group"] = questions["id"].map(group_of)
groups = questions.groupby("group")["id"].apply(list).to_dict()
print("Follow-up pairs:", {g: ids for g, ids in groups.items() if len(ids) > 1})

## 2. Choose the test questions
In this order: the fixed `val` questions (rule 5), one question of each safety category (rule 2), one of each use case (rule 6), then random groups until one third is reached (rule 1).

In [ ]:
SAFETY = ["out of scope", "insufficient evidence", "legal decision", "adversarial"]
USED_FOR_TUNING = ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]
use_case = lambda text: (re.findall(r"3\(([a-d])\)", text) or ["?"])[0]      # "Annex III 3(b)" -> "b"
questions["case"] = questions["use_case"].map(use_case)

val_groups = {group_of[q] for q in USED_FOR_TUNING if q in group_of}
test_groups = set()
free = lambda: sorted(g for g in groups if g not in val_groups | test_groups)

def take(condition, reason):
    """Move one random free group that has a question matching the condition to test."""
    options = [g for g in free() if any(condition(r) for _, r in questions[questions["group"] == g].iterrows())]
    if not options:
        print("  no free question for:", reason)
        return
    test_groups.add(random.choice(options))

for kind in SAFETY:                                                       # rule 2
    if not any(questions.loc[questions["group"].isin(test_groups), "type"] == kind):
        take(lambda r: r["type"] == kind, kind)
for case in "abcd":                                                       # rule 6
    if not any(questions.loc[questions["group"].isin(test_groups), "case"] == case):
        take(lambda r: r["case"] == case, "use case " + case)
while (questions["group"].isin(test_groups)).sum() < math.ceil(len(questions) / 3):   # rule 1
    test_groups.add(random.choice(free()))

questions["split"] = questions["group"].isin(test_groups).map({True: "test", False: "val"})
for name in ["val", "test"]:
    print(name + ":", ", ".join(questions.loc[questions["split"] == name, "id"]))

## 3. Check the rules

In [ ]:
test = questions[questions["split"] == "test"]
checks = {
    "1. at least one third test": len(test) >= len(questions) / 3,
    "2. every safety category in test": all((test["type"] == k).any() for k in SAFETY),
    "3. follow-ups in the same split": all(questions.loc[questions["id"].isin(ids), "split"].nunique() == 1 for ids in groups.values()),
    "4. unique IDs": questions["id"].is_unique,
    "5. B-01 to B-06 in val": set(USED_FOR_TUNING) <= set(questions.loc[questions["split"] == "val", "id"]),
    "6. every use case in test": all((test["case"] == c).any() for c in "abcd"),
}
for rule, ok in checks.items():
    print("OK  " if ok else "NO  ", rule)
print(f"\nval {(questions['split'] == 'val').sum()}, test {len(test)} of {len(questions)}")
pd.crosstab(questions["case"], questions["split"])

**If a rule says NO:** the set does not have such a question yet (for example no adversarial question). Ask the author to add it, then run the notebook again. Do not save until every rule says OK.

## 4. Save and freeze the test split
The `split` column is written into `eval-set-v1.csv`, and the two splits are saved as separate files. The SHA-256 hash of the test file is printed: write it in the Jira comment of KAN-53, so that any later change to the test questions can be seen.

In [ ]:
assert all(checks.values()), "Fix the rules marked NO before saving."
columns = [c for c in questions.columns if c not in ("group", "case")]
questions[columns].to_csv(FOLDER + "eval-set-v1.csv", index=False, quoting=csv.QUOTE_MINIMAL)
for name in ["val", "test"]:
    questions.loc[questions["split"] == name, columns].to_csv(FOLDER + f"eval-set-v1-{name}.csv", index=False)
print("Saved. SHA-256 of the frozen test file:", hashlib.sha256(open(FOLDER + "eval-set-v1-test.csv", "rb").read()).hexdigest())